In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install transformers accelerate bitsandbytes sentence-transformers faiss-cpu \
               beautifulsoup4 lxml pypdf tqdm

**1) Installation des dépendances**

In [2]:
# Installation des librairies optimisées pour le fine-tuning
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Install xformers from PyTorch index for CUDA 12.6 to avoid build errors
!pip install xformers==0.0.33.post2 --index-url https://download.pytorch.org/whl/cu126

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zcnyhnep/unsloth_6eb5333313374f4290f8579c87928718
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zcnyhnep/unsloth_6eb5333313374f4290f8579c87928718
  Resolved https://github.com/unslothai/unsloth.git to commit f3f9d148874389996636e4fd110c7255b9f62c77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Looking in indexes: https://download.pytorch.org/whl/cu126
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 MB 791.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.3 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.5.0
    Uninstalling triton-3.5.0:
      Successfully uninstalled triton-3.5.0
  Attempting uninstall: to

**2) Chargement du Modèle de Base (Llama-3)**

In [ ]:
import os, re
from bs4 import BeautifulSoup

def clean_text(s: str) -> str:
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def split_by_headings(html: str):
    """
    Retourne une liste de blocs: [{"section": "...", "text": "..."}]
    On utilise h1/h2/h3 comme repères de section.
    """
    soup = BeautifulSoup(html, "lxml")

    # Enlever scripts/styles
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Si pas de headings, on prend tout le texte
    headings = soup.find_all(["h1", "h2", "h3"])
    if not headings:
        text = clean_text(soup.get_text(" "))
        return [{"section": "Document", "text": text}] if text else []

    blocks = []
    current_section = None
    current_text = []

    # Parcours linéaire du body
    body = soup.body if soup.body else soup
    for el in body.descendants:
        if getattr(el, "name", None) in ["h1", "h2", "h3"]:
            # flush bloc précédent
            if current_section and current_text:
                txt = clean_text(" ".join(current_text))
                if txt:
                    blocks.append({"section": current_section, "text": txt})
            # nouvelle section
            current_section = clean_text(el.get_text(" "))
            current_text = []
        elif getattr(el, "name", None) in ["p", "li"]:
            t = clean_text(el.get_text(" "))
            if t:
                current_text.append(t)

    # flush final
    if current_section and current_text:
        txt = clean_text(" ".join(current_text))
        if txt:
            blocks.append({"section": current_section, "text": txt})

    return blocks

In [ ]:
# reconstruire la liste html à partir de BASE_PATH
html_files = []
pdf_files = []

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        low = file.lower()
        if low.endswith(".html") or low.endswith(".htm"):
            html_files.append(os.path.join(root, file))
        elif low.endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

print("HTML :", len(html_files))
print("PDF  :", len(pdf_files))

# Parser HTML -> docs structurés
html_docs = []
for path in sorted(html_files):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    blocks = split_by_headings(html)
    src = f"local://{os.path.basename(path)}"  # lien logique
    for b in blocks:
        html_docs.append({
            "text": b["text"],
            "source": src,
            "section": b["section"],
            "page": None,
        })

print("Nombre de blocs HTML (sections) :", len(html_docs))
print("\nExemple :")
print(html_docs[0]["source"], " | ", html_docs[0]["section"])
print(html_docs[0]["text"][:400], "...")

In [ ]:
from pypdf import PdfReader

pdf_docs = []
for path in sorted(pdf_files):
    reader = PdfReader(path)
    src = f"local://{os.path.basename(path)}"
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = clean_text(text)
        if text:
            pdf_docs.append({
                "text": text,
                "source": src,
                "section": f"Page {i}",
                "page": i,
            })

print("Nombre de pages PDF extraites :", len(pdf_docs))
print("\nExemple PDF :")
print(pdf_docs[0]["source"], " | ", pdf_docs[0]["section"])
print(pdf_docs[0]["text"][:400], "...")

In [ ]:
all_docs = html_docs + pdf_docs
print("Docs (blocs) totaux :", len(all_docs))

def chunk_text(text, chunk_size=1200, overlap=200):
    """
    Chunking simple par caractères.
    chunk_size/overlap sont en caractères (MVP).
    """
    text = text.strip()
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

chunks = []
for d in all_docs:
    for c in chunk_text(d["text"], chunk_size=1200, overlap=200):
        chunks.append({
            "text": c,
            "source": d["source"],
            "section": d["section"],
            "page": d["page"],
        })

print("Nombre total de chunks :", len(chunks))
print("\nExemple chunk :")
print(chunks[0]["source"], "|", chunks[0]["section"])
print(chunks[0]["text"][:400], "...")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from tqdm import tqdm

embed_model_name = "intfloat/multilingual-e5-base"
embedder = SentenceTransformer(embed_model_name)

def embed_passages(texts, batch_size=64):
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = ["passage: " + t for t in texts[i:i+batch_size]]
        v = embedder.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        vecs.append(v)
    return np.vstack(vecs).astype("float32")

passage_texts = [c["text"] for c in chunks]
embeddings = embed_passages(passage_texts, batch_size=64)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity via normalized vectors + inner product
index.add(embeddings)

print("FAISS index size :", index.ntotal, "| dim :", dim)

In [ ]:
def retrieve(query, k=6):
    q = embedder.encode(["query: " + query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        c = chunks[int(idx)]
        results.append({
            "score": float(score),
            "text": c["text"],
            "source": c["source"],
            "section": c["section"],
            "page": c["page"],
        })
    return results

# Test
test_q = "Quelles sont les conditions pour candidater au concours CNRS ?"
res = retrieve(test_q, k=5)
for r in res:
    print(f"- {r['score']:.3f} | {r['source']} | {r['section']}")

In [ ]:
!pip -q uninstall -y huggingface_hub
!pip -q install "huggingface_hub>=0.34.0,<1.0"
!pip -q install -U "transformers==4.57.3" accelerate bitsandbytes

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("✅ Gemma chargé:", model_id)

In [ ]:
@torch.inference_mode()
def generate_chat(messages, max_new_tokens=350, temperature=0.2, top_p=0.9):
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()



def format_context(passages):
    lines = []
    for i, p in enumerate(passages, start=1):
        excerpt = p["text"].strip()

        # Extrait plus court pour réduire le bruit
        if len(excerpt) > 450:
            excerpt = excerpt[:450] + "..."

        lines.append(
            f"[{i}] ({p['source']} — {p['section']})\n{excerpt}"
        )
    return "\n\n".join(lines)

INSTRUCTIONS = """INSTRUCTIONS (à respecter):
- Réponds en français.
- Utilise UNIQUEMENT le CONTEXTE.
- Dans le texte, ajoute des citations sous la forme [1], [2], etc. (les numéros correspondent aux extraits du CONTEXTE).
- Si le contexte ne donne pas une réponse complète, réponds quand même avec ce que tu peux, puis ajoute une section "Limites" (1-2 lignes).
- NE DOIS PAS écrire une section "Sources" (ni le mot "Sources"). Elle sera ajoutée automatiquement.
- Quand tu listes des conditions, reformule en puces courtes (sans guillemets) tout en gardant les citations [n].
"""
import re

def strip_model_sources(text: str) -> str:
    # Supprime toute section "Sources" écrite par le modèle (Sources:, Source:, etc.)
    text = re.split(r"\n\s*(Sources?|Source)\s*:\s*\n", text, maxsplit=1, flags=re.IGNORECASE)[0]
    # Supprime un "Sources:" restant sur une ligne
    text = re.sub(r"\n\s*(Sources?|Source)\s*:\s*$", "", text, flags=re.IGNORECASE)
    # Supprime crochets isolés en fin de texte (ex: "[]", "[ ]")
    text = re.sub(r"\n?\[\s*\]\s*$", "", text)
    return text.strip()

def build_sources_from_passages(passages, max_items=8):
    # on garde l'ordre et on supprime les doublons
    seen = set()
    items = []
    for i, p in enumerate(passages, start=1):
        key = (p["source"], p["section"])
        if key not in seen:
            seen.add(key)
            items.append((i, p["source"], p["section"]))
        if len(items) >= max_items:
            break

    lines = ["\nSources:"]
    for i, src, sec in items:
        lines.append(f"- [{i}] {src} | Section: {sec}")
    return "\n".join(lines)

def build_sources_used_only(answer_text: str, passages, max_items=10):
    used = sorted(set(int(n) for n in re.findall(r"\[(\d+)\]", answer_text) if n.isdigit()))
    lines = ["Sources:"]
    count = 0
    for n in used:
        if 1 <= n <= len(passages):
            p = passages[n-1]
            lines.append(f"- [{n}] {p['source']} | Section: {p['section']}")
            count += 1
            if count >= max_items:
                break
    return "\n".join(lines)

def simple_extractive_fallback(passages, max_chars=900):
    # MVP: on renvoie les meilleurs extraits plutôt que rien
    out = ["Je ne peux pas formuler une réponse complète, mais voici les extraits les plus pertinents du contexte :\n"]
    used = []
    for i, p in enumerate(passages[:5], start=1):
        txt = p["text"].strip()
        if len(txt) > max_chars:
            txt = txt[:max_chars] + "..."
        out.append(f"[{i}] ({p['source']} — {p['section']})\n{txt}\n")
        used.append((p["source"], p["section"]))
    out.append("Sources:")
    for s, sec in dict.fromkeys(used):  # unique en gardant l'ordre
        out.append(f"- {s} | Section: {sec}")
    return "\n".join(out)

chat_history = []

def ask(question, k=12):
    global chat_history

    passages = retrieve(question, k=k)
    context = format_context(passages)
    short_history = chat_history[-6:]

    messages = []
    messages.extend(short_history)

    user_msg = f"""{INSTRUCTIONS}

QUESTION:
{question}

CONTEXTE:
{context}
"""
    messages.append({"role": "user", "content": user_msg})

    answer = generate_chat(messages)

    # fallback si refus
    if "je n'ai pas trouvé" in answer.lower():
        final = simple_extractive_fallback(passages)
    else:
        answer = strip_model_sources(answer)
        final = answer + "\n" + build_sources_used_only(answer, passages)

    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": final})

    return final, passages

In [ ]:
passages = retrieve("Quelles sont les conditions pour candidater au concours CNRS ?", k=10)
for i,p in enumerate(passages,1):
    print(f"\n--- [{i}] {p['source']} | {p['section']} (score={p['score']:.3f}) ---")
    print(p["text"][:800])

In [ ]:
ans, passages = ask("Quelles sont les conditions pour candidater au concours CNRS ?", k=10)
print(ans)

In [ ]:
def multi_retrieve(question, k=6):
    variants = [
        question,
        "conditions d'éligibilité concours CNRS",
        "recevabilité candidature concours CNRS",
        "modalités de candidature concours CNRS",
        "inscription dossier de candidature concours CNRS",
    ]
    pool = []
    seen = set()

    for v in variants:
        for r in retrieve(v, k=k):
            key = (r["source"], r["section"], r["text"][:80])
            if key not in seen:
                seen.add(key)
                pool.append(r)

    # tri par score (FAISS)
    pool.sort(key=lambda x: x["score"], reverse=True)
    return pool[:12]

In [ ]:
passages = multi_retrieve("Quelles sont les conditions pour candidater au concours CNRS ?", k=6)
for i,p in enumerate(passages,1):
    print(f"- [{i}] {p['score']:.3f} | {p['source']} | {p['section']}")

In [ ]:
!pip -q install rank-bm25

from rank_bm25 import BM25Okapi

def tokenize_fr(s: str):
    # tokenizer ultra simple (MVP)
    import re
    return re.findall(r"\w+", s.lower())

bm25_corpus = [tokenize_fr(c["text"]) for c in chunks]
bm25 = BM25Okapi(bm25_corpus)

def bm25_retrieve(query, k=12):
    toks = tokenize_fr(query)
    scores = bm25.get_scores(toks)
    top_idx = np.argsort(scores)[::-1][:k]
    res = []
    for idx in top_idx:
        c = chunks[int(idx)]
        res.append({
            "score": float(scores[idx]),
            "text": c["text"],
            "source": c["source"],
            "section": c["section"],
            "page": c["page"],
        })
    return res

In [ ]:
def hybrid_retrieve(question, k_faiss=6, k_bm25=6):
    fa = multi_retrieve(question, k=k_faiss)
    bm = bm25_retrieve(question, k=k_bm25)

    # fusion
    pool = []
    seen = set()
    for r in fa + bm:
        key = (r["source"], r["section"], r["text"][:80])
        if key not in seen:
            seen.add(key)
            pool.append(r)

    return pool[:12]

In [ ]:
passages = hybrid_retrieve("Quelles sont les conditions pour candidater au concours CNRS ?")
for i,p in enumerate(passages,1):
    print(f"\n--- [{i}] {p['source']} | {p['section']} (score={p['score']:.3f}) ---")
    print(p["text"][:800])

In [ ]:
ans, passages = ask("Quelles sont les conditions pour candidater au concours CNRS ?")
print(ans)

In [ ]:
def chat_loop():
    print("=== Chatbot RAG (CNRS) ===")
    print("Commandes: /quit (sortir), /reset (vider l'historique)\n")

    while True:
        q = input("Vous: ").strip()
        if not q:
            continue

        if q.lower() in ["/quit", "quit", "exit"]:
            print("Fin du chat.")
            break

        if q.lower() in ["/reset", "reset"]:
            global chat_history
            chat_history = []
            print("✅ Historique réinitialisé.\n")
            continue

        ans, _ = ask(q)
        print("\nAssistant:\n" + ans + "\n")

# Lancer le chat
chat_loop()